In [1]:
import pandas as pd 
import numpy as np
import os, sys
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.impute import SimpleImputer
from pathlib import Path
from data_preprocessing import radiomics_load, train_test_split, var_and_corr_filtering,expand_features, preprocess_test, preprocess_train, PCA_formatting

# Data Loading & Analysis

In [3]:
data = Path("/projects/net_contrast_classification/contrast_phase/data/cleaned_data_1.csv")

features_dir = Path("/projects/net_contrast_classification/contrast_phase/Radiomics/features")

radiomics = radiomics_load(data, features_dir)
radiomics

,image,SubjectKeyRadiology,phase_timing,contrast,organ,original_firstorder_10Percentile,original_firstorder_90Percentile,original_firstorder_Energy,original_firstorder_Entropy,original_firstorder_InterquartileRange,...,original_glcm_Correlation,original_glcm_Autocorrelation,original_glcm_ClusterTendency,original_glcm_DifferenceEntropy,original_glszm_ZoneEntropy,original_glszm_SmallAreaEmphasis,original_glszm_GrayLevelNonUniformity,original_glszm_SizeZoneNonUniformity,original_gldm_SmallDependenceEmphasis,original_gldm_DependenceNonUniformity
0,NKI-d23231-00-0510_20190509_kalina99_3510.nii.gz,NKI-d23231-00-0510,Just Right,Arterial,spleen,73.0,129.0,1.411073e+09,2.345055,28.0,...,0.331391,161.467821,4.317435,1.807917,5.579062,0.607571,789.747481,2234.026508,0.057494,8059.752032
1,NKI-d23231-00-0510_20190509_kalina99_3510.nii.gz,NKI-d23231-00-0510,Just Right,Arterial,kidney_right,35.0,174.0,2.051722e+09,3.368361,96.0,...,0.573631,172.405745,24.124352,2.463745,5.972511,0.623694,1069.817432,4300.504964,0.100165,11505.342655
2,NKI-d23231-00-0510_20190509_kalina99_3510.nii.gz,NKI-d23231-00-0510,Just Right,Arterial,kidney_left,29.0,175.0,1.875893e+09,3.462463,101.0,...,0.625287,168.789858,27.864372,2.459546,5.964703,0.651090,921.112624,4496.306967,0.102064,10496.241509
3,NKI-d23231-00-0510_20190509_kalina99_3510.nii.gz,NKI-d23231-00-0510,Just Right,Arterial,gallbladder,15.0,92.0,6.779767e+06,2.671679,44.0,...,0.277793,49.512989,6.008309,2.088526,5.214034,0.571997,22.650794,58.957672,0.102627,169.717503
4,NKI-d23231-00-0510_20190509_kalina99_3510.nii.gz,NKI-d23231-00-0510,Just Right,Arterial,liver,57.0,111.0,9.182282e+09,2.273400,28.0,...,0.304811,769.295507,3.639820,1.789879,5.516856,0.589709,7598.796877,15753.440790,0.049232,79966.681546
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
166655,NKI-d23231-00-0521_20180208_kalina99_3587.nii.gz,NKI-d23231-00-0521,Just Right,Portal,portal_vein_and_splenic_vein,94.0,199.0,3.244518e+08,2.985268,54.0,...,0.441004,162.569181,11.146397,2.231316,5.916112,0.579998,180.277874,546.459272,0.132850,1009.163186
166656,NKI-d23231-00-0521_20180208_kalina99_3587.nii.gz,NKI-d23231-00-0521,Just Right,Portal,iliac_artery_left,47.0,169.0,1.198564e+08,3.299687,60.0,...,0.538260,169.284619,19.404490,2.269254,6.271613,0.553009,87.013220,307.181303,0.143030,577.424611
166657,NKI-d23231-00-0521_20180208_kalina99_3587.nii.gz,NKI-d23231-00-0521,Just Right,Portal,iliac_artery_right,47.0,167.0,2.443429e+08,3.426126,67.0,...,0.574321,193.314195,96.147512,2.481168,6.542065,0.607940,99.914074,464.976296,0.155215,700.690310
166658,NKI-d23231-00-0521_20180208_kalina99_3587.nii.gz,NKI-d23231-00-0521,Just Right,Portal,iliac_vena_left,26.0,122.0,1.316743e+08,3.136003,42.0,...,0.626491,113.830952,18.455433,2.138249,6.222564,0.600864,131.003509,578.122807,0.110335,1047.527077


In [4]:
radiomics[radiomics.original_firstorder_Energy.isna()].organ.value_counts()

organ
superior_vena_cava              7056
pulmonary_vein                  4901
iliac_vena_left                 1788
gallbladder                     1686
iliac_vena_right                1660
iliac_artery_right              1254
iliac_artery_left               1207
kidney_right                      74
spleen                            67
adrenal_gland_left                58
adrenal_gland_right               55
heart                             50
small_bowel                       42
portal_vein_and_splenic_vein      38
kidney_left                       36
pancreas                          19
stomach                           14
inferior_vena_cava                 8
aorta                              6
liver                              5
Name: count, dtype: int64

In [5]:
radiomics[radiomics.original_glszm_SizeZoneNonUniformity.isna()].organ.value_counts()

organ
superior_vena_cava              7056
pulmonary_vein                  4901
iliac_vena_left                 1788
gallbladder                     1686
iliac_vena_right                1660
iliac_artery_right              1254
iliac_artery_left               1207
kidney_right                      74
spleen                            67
adrenal_gland_left                58
adrenal_gland_right               55
heart                             50
small_bowel                       42
portal_vein_and_splenic_vein      38
kidney_left                       36
pancreas                          19
stomach                           14
inferior_vena_cava                 8
aorta                              6
liver                              5
Name: count, dtype: int64

In [6]:
len(radiomics[radiomics.contrast == '0'])

0

In [7]:
radiomics = radiomics[radiomics.contrast != '0']
radiomics.contrast.value_counts()

contrast
Portal          74160
Arterial        49740
Non-contrast    42760
Name: count, dtype: int64

In [9]:
data

,NiiFile,SegNiiFile,Original File,Destination Folder,New File,SubjectKeyRadiology,ExamDate,file,is_liver_imaged,contrast,...,ImageOrientationPatientDICOM,ConversionSoftware,ConversionSoftwareVersion,BodyPartExamined,RawImage,SliceThickness,AcquisitionTime_sec,DICOM_phase,MatchKey,exist_on_server
0,Z:\_archived\DICOM-batch1-kalina01\NET_0000_00...,Z:\_archived\DICOM-batch1-kalina01\NET_0000_00...,../DICOM-batch1\NKI-d23231-00-0063\20120416 CT...,DICOM-batch1-kalina01,NET_0000_0000.nii.gz,NKI-d23231-00-0063,2012-04-16,NET_0000_0000.nii.gz,Partially,Arterial,...,"[1, 0, 0, 0, 1, 0]",dcm2niix,v1.0.20230411,ABDOMEN,NaN,1.0,40231.70000,Arterial,NKI-d23231-00-0063_20120416_kalina01_0000.nii.gz,\\nki.nl\res\RD CRC-data\_archive\IRBd23231-AM...
1,Z:\_archived\DICOM-batch1-kalina01\NET_0001_00...,Z:\_archived\DICOM-batch1-kalina01\NET_0001_00...,../DICOM-batch1\NKI-d23231-00-0070\20150821 CT...,DICOM-batch1-kalina01,NET_0001_0000.nii.gz,NKI-d23231-00-0070,2015-08-21,NET_0001_0000.nii.gz,Yes,Non-contrast,...,"[1, 0, 0, 0, 1, 0]",dcm2niix,v1.0.20230411,LEVER,NaN,1.0,30853.90000,Non-contrast,NKI-d23231-00-0070_20150821_kalina01_0001.nii.gz,NaN
2,Z:\_archived\DICOM-batch1-kalina01\NET_0002_00...,Z:\_archived\DICOM-batch1-kalina01\NET_0002_00...,../DICOM-batch1\NKI-d23231-00-0036\20140218 CT...,DICOM-batch1-kalina01,NET_0002_0000.nii.gz,NKI-d23231-00-0036,2014-02-18,NET_0002_0000.nii.gz,Yes,Portal,...,"[1, 0, 0, 0, 1, 0]",dcm2niix,v1.0.20230411,ABDOMEN,NaN,1.0,32599.60000,NaN,NKI-d23231-00-0036_20140218_kalina01_0002.nii.gz,\\nki.nl\res\RD CRC-data\_archive\IRBd23231-AM...
3,Z:\_archived\DICOM-batch1-kalina01\NET_0003_00...,Z:\_archived\DICOM-batch1-kalina01\NET_0003_00...,../DICOM-batch1\NKI-d23231-00-0071\20130924 CT...,DICOM-batch1-kalina01,NET_0003_0000.nii.gz,NKI-d23231-00-0071,2013-09-24,NET_0003_0000.nii.gz,Yes,Arterial,...,"[1, 0, 0, 0, 1, 0]",dcm2niix,v1.0.20230411,ABDOMEN,NaN,1.5,45013.30692,NaN,NKI-d23231-00-0071_20130924_kalina01_0003.nii.gz,\\nki.nl\res\RD CRC-data\_archive\IRBd23231-AM...
4,Z:\_archived\DICOM-batch1-kalina01\NET_0004_00...,Z:\_archived\DICOM-batch1-kalina01\NET_0004_00...,../DICOM-batch1\NKI-d23231-00-0077\20141222 CT...,DICOM-batch1-kalina01,NET_0004_0000.nii.gz,NKI-d23231-00-0077,2014-12-22,NET_0004_0000.nii.gz,Yes,Portal,...,"[1, 0, 0, 0, 1, 0]",dcm2niix,v1.0.20230411,ABDOMEN,NaN,1.5,51587.10418,Portal,NKI-d23231-00-0077_20141222_kalina01_0004.nii.gz,\\nki.nl\res\RD CRC-data\_archive\IRBd23231-AM...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8729,Z:\_archived\DICOM-batch3-kalina99\NET_5827_00...,Z:\_archived\DICOM-batch3-kalina99\NET_5827_00...,../DICOM-batch3\NKI-d23231-00-0888\20220420 CT...,DICOM-batch3-kalina99,NET_5827_0000.nii.gz,NKI-d23231-00-0888,2022-04-20,NET_5827_0000.nii.gz,Yes,Portal,...,"[1, 0, 0, 0, 1, 0]",dcm2niix,v1.0.20230411,ABDOMEN,NaN,1.5,54796.11932,Portal,NKI-d23231-00-0888_20220420_kalina99_5827.nii.gz,\\nki.nl\res\RD CRC-data\_archive\IRBd23231-AM...
8730,Z:\_archived\DICOM-batch3-kalina99\NET_5828_00...,Z:\_archived\DICOM-batch3-kalina99\NET_5828_00...,../DICOM-batch3\NKI-d23231-00-0888\20230424 CT...,DICOM-batch3-kalina99,NET_5828_0000.nii.gz,NKI-d23231-00-0888,2023-04-24,NET_5828_0000.nii.gz,Yes,Arterial,...,"[1, 0, 0, 0, 1, 0]",dcm2niix,v1.0.20230411,ABDOMEN,NaN,1.0,43545.56100,Arterial,NKI-d23231-00-0888_20230424_kalina99_5828.nii.gz,\\nki.nl\res\RD CRC-data\_archive\IRBd23231-AM...
8731,Z:\_archived\DICOM-batch3-kalina99\NET_5829_00...,Z:\_archived\DICOM-batch3-kalina99\NET_5829_00...,../DICOM-batch3\NKI-d23231-00-0888\20230424 CT...,DICOM-batch3-kalina99,NET_5829_0000.nii.gz,NKI-d23231-00-0888,2023-04-24,NET_5829_0000.nii.gz,Yes,Portal,...,"[1, 0, 0, 0, 1, 0]",dcm2niix,v1.0.20230411,ABDOMEN,NaN,1.0,43580.13800,NaN,NKI-d23231-00-0888_20230424_kalina99_5829.nii.gz,\\nki.nl\res\RD CRC-data\_archive\IRBd23231-AM...
8732,Z:\_archived\DICOM-batch3-kalina99\NET_5830_00...,Z:\_archived\DICOM-batch3-kalina99\NET_5830_00...,../DICOM-batch3\NK

In [56]:
display(data.contrast.value_counts(), f"Null values: {int(data.contrast.isna().sum())}",
        data.is_liver_imaged.value_counts(), f"Null values: {int(data.is_liver_imaged.isna().sum())}",
        data.phase_timing.value_counts(), f"Null values: {int(data.phase_timing.isna().sum())}",
        data.is_lesionfree.value_counts(), f"Null values: {int(data.is_lesionfree.isna().sum())}")

contrast
Portal          3708
Arterial        2488
Non-contrast    2119
0                419
Name: count, dtype: int64

'Null values: 0'

is_liver_imaged
Yes          7866
Partially     377
0             330
Name: count, dtype: int64

'Null values: 161'

phase_timing
Just Right      5110
Non-contrast    2119
Too Early        916
0.0              416
Too Late         173
Name: count, dtype: int64

'Null values: 0'

is_lesionfree
No     6587
Yes    2147
Name: count, dtype: int64

'Null values: 0'

In [58]:
radiomics_merged = radiomics.merge(data[['phase_timing', 'MatchKey', 'SubjectKeyRadiology']], left_on = 'image', right_on= 'MatchKey', how = 'left')
radiomics_merged = radiomics_merged.drop("MatchKey", axis=1)

col = radiomics_merged.pop('SubjectKeyRadiology')
radiomics_merged.insert(1, 'SubjectKeyRadiology', col)

col = radiomics_merged.pop('phase_timing')
radiomics_merged.insert(2, 'phase_timing', col)

radiomics_merged

,image,SubjectKeyRadiology,phase_timing,contrast,organ,original_firstorder_10Percentile,original_firstorder_90Percentile,original_firstorder_Energy,original_firstorder_Entropy,original_firstorder_InterquartileRange,...,original_glcm_Correlation,original_glcm_Autocorrelation,original_glcm_ClusterTendency,original_glcm_DifferenceEntropy,original_glszm_ZoneEntropy,original_glszm_SmallAreaEmphasis,original_glszm_GrayLevelNonUniformity,original_glszm_SizeZoneNonUniformity,original_gldm_SmallDependenceEmphasis,original_gldm_DependenceNonUniformity
0,NKI-d23231-00-0510_20190509_kalina99_3510.nii.gz,NKI-d23231-00-0510,Just Right,Arterial,spleen,73.0,129.0,1.411073e+09,2.345055,28.0,...,0.331391,161.467821,4.317435,1.807917,5.579062,0.607571,789.747481,2234.026508,0.057494,8059.752032
1,NKI-d23231-00-0510_20190509_kalina99_3510.nii.gz,NKI-d23231-00-0510,Just Right,Arterial,kidney_right,35.0,174.0,2.051722e+09,3.368361,96.0,...,0.573631,172.405745,24.124352,2.463745,5.972511,0.623694,1069.817432,4300.504964,0.100165,11505.342655
2,NKI-d23231-00-0510_20190509_kalina99_3510.nii.gz,NKI-d23231-00-0510,Just Right,Arterial,kidney_left,29.0,175.0,1.875893e+09,3.462463,101.0,...,0.625287,168.789858,27.864372,2.459546,5.964703,0.651090,921.112624,4496.306967,0.102064,10496.241509
3,NKI-d23231-00-0510_20190509_kalina99_3510.nii.gz,NKI-d23231-00-0510,Just Right,Arterial,gallbladder,15.0,92.0,6.779767e+06,2.671679,44.0,...,0.277793,49.512989,6.008309,2.088526,5.214034,0.571997,22.650794,58.957672,0.102627,169.717503
4,NKI-d23231-00-0510_20190509_kalina99_3510.nii.gz,NKI-d23231-00-0510,Just Right,Arterial,liver,57.0,111.0,9.182282e+09,2.273400,28.0,...,0.304811,769.295507,3.639820,1.789879,5.516856,0.589709,7598.796877,15753.440790,0.049232,79966.681546
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
166655,NKI-d23231-00-0521_20180208_kalina99_3587.nii.gz,NKI-d23231-00-0521,Just Right,Portal,portal_vein_and_splenic_vein,94.0,199.0,3.244518e+08,2.985268,54.0,...,0.441004,162.569181,11.146397,2.231316,5.916112,0.579998,180.277874,546.459272,0.132850,1009.163186
166656,NKI-d23231-00-0521_20180208_kalina99_3587.nii.gz,NKI-d23231-00-0521,Just Right,Portal,iliac_artery_left,47.0,169.0,1.198564e+08,3.299687,60.0,...,0.538260,169.284619,19.404490,2.269254,6.271613,0.553009,87.013220,307.181303,0.143030,577.424611
166657,NKI-d23231-00-0521_20180208_kalina99_3587.nii.gz,NKI-d23231-00-0521,Just Right,Portal,iliac_artery_right,47.0,167.0,2.443429e+08,3.426126,67.0,...,0.574321,193.314195,96.147512,2.481168,6.542065,0.607940,99.914074,464.976296,0.155215,700.690310
166658,NKI-d23231-00-0521_20180208_kalina99_3587.nii.gz,NKI-d23231-00-0521,Just Right,Portal,iliac_vena_left,26.0,122.0,1.316743e+08,3.136003,42.0,...,0.626491,113.830952,18.455433,2.138249,6.222564,0.600864,131.003509,578.122807,0.110335,1047.527077


# Splitting

In [59]:
train_data, test_data = train_test_split(radiomics_merged, test_size=0.3, grouping="SubjectKeyRadiology")

In [17]:
from sklearn.model_selection import GroupShuffleSplit

splitter = GroupShuffleSplit(test_size=0.3,random_state=42)

train_idx, test_idx = next(splitter.split(radiomics_merged, groups=radiomics_merged["SubjectKeyRadiology"]))

train_data = radiomics_merged.iloc[train_idx]
test_data = radiomics_merged.iloc[test_idx]

In [60]:
train_ids = train_data.image.apply(lambda x: x.split("_")[0])
test_ids = test_data.image.apply(lambda x: x.split("_")[0])

len(set(train_ids) & set(test_ids))

0

In [61]:
train_data

,image,SubjectKeyRadiology,phase_timing,contrast,organ,original_firstorder_10Percentile,original_firstorder_90Percentile,original_firstorder_Energy,original_firstorder_Entropy,original_firstorder_InterquartileRange,...,original_glcm_Correlation,original_glcm_Autocorrelation,original_glcm_ClusterTendency,original_glcm_DifferenceEntropy,original_glszm_ZoneEntropy,original_glszm_SmallAreaEmphasis,original_glszm_GrayLevelNonUniformity,original_glszm_SizeZoneNonUniformity,original_gldm_SmallDependenceEmphasis,original_gldm_DependenceNonUniformity
160,NKI-d23231-00-0515_20160707_kalina99_3527.nii.gz,NKI-d23231-00-0515,Too Early,Arterial,spleen,50.5,113.0,1.595007e+09,2.499157,31.0,...,0.331934,246.773797,5.151847,1.927021,5.692982,0.598227,1450.030454,4185.030695,0.065461,14333.585314
161,NKI-d23231-00-0515_20160707_kalina99_3527.nii.gz,NKI-d23231-00-0515,Too Early,Arterial,kidney_right,37.5,156.0,1.439099e+09,3.204356,68.0,...,0.565912,164.761755,16.464644,2.268358,5.822248,0.626789,773.095957,3254.095844,0.091382,9549.153283
162,NKI-d23231-00-0515_20160707_kalina99_3527.nii.gz,NKI-d23231-00-0515,Too Early,Arterial,kidney_left,35.5,152.0,1.365023e+09,3.185311,66.5,...,0.571313,161.377828,15.941544,2.245522,5.849100,0.615520,785.171590,3134.187550,0.088984,9105.674526
163,NKI-d23231-00-0515_20160707_kalina99_3527.nii.gz,NKI-d23231-00-0515,Too Early,Arterial,gallbladder,-9.0,47.0,1.193202e+07,2.259217,29.0,...,0.299790,41.665865,3.310482,1.748083,5.147911,0.588907,113.761769,227.005706,0.059292,974.997175
164,NKI-d23231-00-0515_20160707_kalina99_3527.nii.gz,NKI-d23231-00-0515,Too Early,Arterial,liver,41.0,93.0,7.637757e+09,2.200781,26.0,...,0.325304,672.405340,3.323770,1.716456,5.388663,0.581572,10487.194332,18484.242544,0.046004,94372.212207
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
166635,NKI-d23231-00-0519_20100701_kalina99_3577.nii.gz,NKI-d23231-00-0519,Just Right,Portal,portal_vein_and_splenic_vein,58.0,151.0,2.372264e+08,2.958910,46.0,...,0.252392,170.121770,8.688480,2.378827,5.813408,0.602019,272.199248,900.775188,0.145196,2015.624865
166636,NKI-d23231-00-0519_20100701_kalina99_3577.nii.gz,NKI-d23231-00-0519,Just Right,Portal,iliac_artery_left,38.0,158.0,5.367264e+08,3.621822,58.0,...,0.540246,264.222508,183.325094,2.865958,6.864171,0.689673,101.267574,969.792744,0.170817,1134.352142
166637,NKI-d23231-00-0519_20100701_kalina99_3577.nii.gz,NKI-d23231-00-0519,Just Right,Portal,iliac_artery_right,43.0,158.0,6.307420e+08,3.581540,53.0,...,0.694872,368.448207,274.461149,2.816499,6.876047,0.690468,103.363069,988.468332,0.172716,1095.834663
166638,NKI-d23231-00-0519_20100701_kalina99_3577.nii.gz,NKI-d23231-00-0519,Just Right,Portal,iliac_vena_left,27.0,105.0,1.521878e+08,2.798249,39.0,...,0.385276,119.895577,7.713320,2.085660,5.851769,0.617828,228.667934,841.953105,0.095377,2076.153510


# Preprocessing

In [66]:
meta_cols = list(train_data.iloc[:, :5].columns)

train_pipeline = preprocess_train(train_data, meta_cols)

train_processed = train_pipeline["data_processed"]
X_train_pca = train_pipeline["X_pca"]
train_processed

,image,SubjectKeyRadiology,phase_timing,contrast,adrenal_gland_left_firstorder_10Percentile,adrenal_gland_left_firstorder_90Percentile,adrenal_gland_left_firstorder_Energy,adrenal_gland_left_firstorder_Entropy,adrenal_gland_left_firstorder_Kurtosis,adrenal_gland_left_firstorder_Minimum,...,stomach_firstorder_Energy,stomach_firstorder_Entropy,stomach_firstorder_Maximum,stomach_firstorder_Median,stomach_firstorder_Minimum,stomach_glcm_Correlation,stomach_glszm_ZoneEntropy,stomach_glszm_SmallAreaEmphasis,stomach_glszm_GrayLevelNonUniformity,stomach_glszm_SizeZoneNonUniformity
0,NKI-d23231-00-0001_20060920_kalina12_0045.nii.gz,NKI-d23231-00-0001,Too Early,Portal,0.308363,0.446967,0.230648,0.362596,-0.242303,0.206969,...,0.859243,0.099530,0.107703,0.285829,0.105089,0.894737,0.592513,-0.626645,0.798799,0.715094
1,NKI-d23231-00-0001_20070608_kalina03_0023.nii.gz,NKI-d23231-00-0001,Just Right,Portal,0.767188,0.581759,0.806248,0.061035,-0.227770,0.589575,...,0.291669,1.079447,0.737064,0.450436,0.105089,0.449950,1.375772,-0.344830,-0.135113,0.433511
2,NKI-d23231-00-0001_20070608_kalina07_0032.nii.gz,NKI-d23231-00-0001,Non-contrast,Non-contrast,-0.821050,-1.339034,-0.857524,-0.684130,-0.145322,-0.247376,...,0.359369,1.009281,0.698185,0.261138,0.105089,0.650629,1.568824,-0.540743,-0.371791,0.110170
3,NKI-d23231-00-0001_20070815_kalina14_0007.nii.gz,NKI-d23231-00-0001,Just Right,Portal,0.978952,-0.159600,0.197907,-1.205143,0.162089,1.665654,...,-0.573139,-1.237660,-0.368570,0.318750,1.186037,-0.210362,0.629883,-1.886213,-0.836511,-1.148070
4,NKI-d23231-00-0001_20070815_kalina99_0000.nii.gz,NKI-d23231-00-0001,Just Right,Portal,0.943658,0.244778,0.775801,-0.413095,0.711832,0.135230,...,-0.460303,-0.284283,-0.174173,0.326981,1.005879,-0.271568,1.287666,-1.755195,-0.956994,-1.073037
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5667,NKI-d23231-00-0888_20220420_kalina99_5827.nii.gz,NKI-d23231-00-0888,Just Right,Portal,0.908364,0.952439,0.314040,0.367358,0.137988,-0.199550,...,-0.308376,0.294838,-0.317540,0.005997,0.105089,0.196472,0.255930,0.114496,-0.262090,-0.163259
5668,NKI-d23231-00-0888_20230424_kalina99_5828.nii.gz,NKI-d23231-00-0888,Just Right,Arterial,0.566011,-0.493211,-1.243993,-0.979812,1.996222,1.306961,...,-0.623066,-0.385355,-0.217912,0.401054,0.105089,-2.561788,-0.739318,-1.743220,-0.300341,-0.953126
5669,NKI-d23231-00-0888_20230424_kalina99_5829.nii.gz,NKI-d23231-00-0888,Just Right,Portal,0.378952,-0.648223,-1.236682,-1.014415,3.009639,1.761306,...,-0.574766,-0.497723,-0.220342,0.384593,0.105089,-1.275965,-0.648691,-0.806422,-0.367169,-0.882091
5670,NKI-d23231-00-0888_20230801_kalina99_5830.nii.gz,NKI-d23231-00-0888,Just Right,Arterial,0.851894,2.462115,-1.284141,0.551022,0.397372,2.885211,...,-0.617343,-0.834574,-0.171743,0.738498,0.305264,-3.205634,-1.370901,-3.655472,-0.224134,-1.198989


In [67]:
test_processed, X_test_pca = preprocess_test(test_data, meta_cols,train_pipeline)
test_processed

,image,SubjectKeyRadiology,phase_timing,contrast,adrenal_gland_left_firstorder_10Percentile,adrenal_gland_left_firstorder_90Percentile,adrenal_gland_left_firstorder_Energy,adrenal_gland_left_firstorder_Entropy,adrenal_gland_left_firstorder_Kurtosis,adrenal_gland_left_firstorder_Minimum,...,stomach_firstorder_Energy,stomach_firstorder_Entropy,stomach_firstorder_Maximum,stomach_firstorder_Median,stomach_firstorder_Minimum,stomach_glcm_Correlation,stomach_glszm_ZoneEntropy,stomach_glszm_SmallAreaEmphasis,stomach_glszm_GrayLevelNonUniformity,stomach_glszm_SizeZoneNonUniformity
0,NKI-d23231-00-0005_20071116_kalina99_0005.nii.gz,NKI-d23231-00-0005,Too Late,Portal,1.084835,-0.496581,-1.055439,-2.368681,0.707978,1.378700,...,-0.416140,-1.242934,-0.538667,-0.018694,0.725633,0.040929,0.257560,-0.812183,-0.627110,-0.796870
1,NKI-d23231-00-0005_20080529_kalina15_0052.nii.gz,NKI-d23231-00-0005,Just Right,Portal,0.802482,0.615457,0.140906,-0.011447,-0.083883,0.565662,...,-0.338127,-0.777555,-0.220342,0.442205,0.105089,0.466769,0.123690,-0.899979,-0.026279,-0.354702
2,NKI-d23231-00-0005_20090128_kalina14_0044.nii.gz,NKI-d23231-00-0005,Non-contrast,Non-contrast,-0.573990,-1.035751,-0.959046,-0.435939,0.273509,-0.462592,...,0.141283,0.929343,-0.106134,0.112991,0.105089,0.549952,1.151392,-0.653082,-0.124328,0.150081
3,NKI-d23231-00-0005_20090128_kalina99_0006.nii.gz,NKI-d23231-00-0005,Just Right,Portal,1.155423,1.222024,0.914424,0.372426,0.018380,0.206969,...,0.050547,0.711792,-0.042954,0.491588,0.105089,0.496860,0.781463,-0.305832,-0.191486,0.054098
4,NKI-d23231-00-0005_20100204_kalina04_0000.nii.gz,NKI-d23231-00-0005,Too Early,Arterial,0.096599,-0.024807,-0.302494,0.131205,-0.047029,-0.247376,...,-0.567021,-0.561935,-0.198472,0.252907,0.105089,-0.644541,-0.410743,-1.453672,-0.357334,-0.949264
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2656,NKI-d23231-00-0887_20230524_kalina99_5818.nii.gz,NKI-d23231-00-0887,Just Right,Portal,0.975423,0.952439,0.907648,0.067953,0.330122,0.661314,...,0.291320,1.237404,-0.412309,-0.133919,0.285247,0.319530,0.636458,1.196215,-0.781273,-0.205495
2657,NKI-d23231-00-0887_20230524_kalina99_5819.nii.gz,NKI-d23231-00-0887,Non-contrast,Non-contrast,-0.009284,-1.069449,-0.846562,-1.156829,1.089184,-0.773459,...,0.268216,0.913490,-0.526517,-0.298526,0.245212,0.400874,0.447591,1.272296,-0.813329,-0.303525
2658,NKI-d23231-00-0887_20230524_kalina99_5820.nii.gz,NKI-d23231-00-0887,Just Right,Arterial,1.190717,1.559005,2.102471,0.588842,0.501455,-0.079986,...,0.273565,1.359204,-0.310251,-0.125689,0.105089,0.309219,0.763150,1.122254,-0.791983,-0.199813
2659,NKI-d23231-00-0887_20230830_kalina99_5821.nii.gz,NKI-d23231-00-0887,Just Right,Arterial,0.378952,0.244778,-0.088426,-0.142538,-0.034794,0.087405,...,-0.466158,0.452780,-0.319970,-0.265605,-0.515455,0.018110,0.241433,-0.075885,-0.469470,-0.405543


In [68]:
train_ids = train_processed.image.apply(lambda x: x.split("_")[0])
test_ids = test_processed.image.apply(lambda x: x.split("_")[0])

len(set(train_ids) & set(test_ids))

0

In [69]:
# Define path
save_dir = "/projects/net_contrast_classification/contrast_phase/preprocessed_data"

# Create directory if it doesn't exist
os.makedirs(save_dir, exist_ok=True)


In [70]:
train_processed.to_csv(f"{save_dir}/train.csv", index=False)
test_processed.to_csv(f"{save_dir}/test.csv", index=False)

# Convert PCA arrays to DataFrames
X_train_pca_df = PCA_formatting(X_train_pca, train_processed)
X_test_pca_df = PCA_formatting(X_test_pca, test_processed)

# Save CSVs
X_train_pca_df.to_csv(f"{save_dir}/train_pca.csv", index=False)
X_test_pca_df.to_csv(f"{save_dir}/test_pca.csv", index=False)
